# 01 — Generated corpus

Read-only view over `data/generations.jsonl`. Run `python gen_data.py` first.

This notebook checks the corpus is what stage 5 assumes it is: every cell full,
lengths in range, and no message naming the emotion it is supposed to be carrying
implicitly.

In [ ]:
import sys, pathlib, json, re

REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
import config, personas, plot

if not config.GENERATIONS_JSONL.exists():
    raise SystemExit(f"{config.GENERATIONS_JSONL} not found — run `python gen_data.py` first")

df = pd.read_json(config.GENERATIONS_JSONL, lines=True)
df["words"] = df["text"].str.split().str.len()

expected = len(personas.PERSONAS) * len(config.EMOTIONS) * config.N_SCENARIOS_PER_CELL
print(f"{len(df)} of {expected} expected generations")
print(f"generator: {sorted(df['generator_model'].unique())}")
df.head(3)[["persona_id", "emotion", "scenario_id", "words"]]

## System prompt lengths

`default_assistant` anchors the assistant axis, so it must not be the shortest
written prompt in the set — if it were, "distance from the anchor" would be partly
"longer prompt", which is the confound `bare_template` exists to detect, applied to
the anchor itself. All written prompts are matched to a narrow band.

`bare_template` sends no system message at all (counted as 0 here); the chat
template supplies its own.

In [ ]:
wc = pd.DataFrame([
    {"intuited_rank": p["distance_rank"],
     "persona": p["id"],
     "words": len(p["system_prompt"].split()) if p["system_prompt"] else 0,
     "role": ("control" if p["is_control"]
              else "ANCHOR" if p["id"] == personas.DEFAULT_PERSONA_ID
              else "persona")}
    for p in personas.PERSONAS
]).set_index("intuited_rank")
display(wc)

written = wc.loc[wc["words"] > 0, "words"]
anchor = int(wc.loc[wc["persona"] == personas.DEFAULT_PERSONA_ID, "words"].iloc[0])
print(f"written prompts: {written.min()}–{written.max()} words "
      f"(spread {written.max() - written.min()})")
print(f"anchor '{personas.DEFAULT_PERSONA_ID}': {anchor} words — "
      + ("not the shortest, good" if anchor > written.min()
         else "THE SHORTEST: length confound sits on the anchor"))

## Coverage

Every cell should hold exactly 30. Anything less means calls failed — rerun
`gen_data.py`, which refetches only the gaps.

In [ ]:
coverage = df.pivot_table(index="persona_id", columns="emotion",
                          values="scenario_id", aggfunc="count").fillna(0).astype(int)
coverage = coverage.reindex([p["id"] for p in personas.PERSONAS])[config.EMOTIONS]

short = (coverage != config.N_SCENARIOS_PER_CELL).sum().sum()
print(f"{short} cell(s) not at {config.N_SCENARIOS_PER_CELL}" if short else
      f"all {coverage.size} cells complete at {config.N_SCENARIOS_PER_CELL}")

# Same 30 topics in every cell — the fully crossed design depends on this.
per_cell = df.groupby(["persona_id", "emotion"])["scenario_id"].apply(frozenset)
print(f"distinct topic sets across cells: {per_cell.nunique()} (must be 1)")

coverage.style.background_gradient(cmap="Blues", vmin=0, vmax=config.N_SCENARIOS_PER_CELL)

## Length

Target is 100–150 words. Persona-driven length differences are expected (the
detective is terse, the tutor is not) and are not themselves a problem — but a
persona pinned at the `max_tokens` ceiling is being truncated mid-sentence, which
would put a systematic artifact in that persona's pooled activations.

In [ ]:
stats = (df.groupby("persona_id")["words"]
           .agg(["count", "mean", "std", "min", "max"])
           .reindex([p["id"] for p in personas.PERSONAS]).round(1))
lo, hi = config.TARGET_WORDS
stats["pct_in_range"] = (df.assign(ok=df["words"].between(lo, hi))
                           .groupby("persona_id")["ok"].mean().mul(100).round(1))
display(stats)

if "completion_tokens" in df.columns and df["completion_tokens"].notna().any():
    at_ceiling = (df["completion_tokens"] >= config.GEN_MAX_TOKENS).sum()
    print(f"{at_ceiling} generation(s) hit the {config.GEN_MAX_TOKENS}-token ceiling "
          f"({at_ceiling / len(df):.1%}) — these are truncated mid-sentence")

In [ ]:
import matplotlib.pyplot as plt

plot.style(plt)
order = [p["id"] for p in personas.PERSONAS]
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.grid(axis="x", visible=False)

for i, pid in enumerate(order):
    w = df.loc[df["persona_id"] == pid, "words"]
    colour = plot.SERIES_CONTROL if personas.PERSONAS_BY_ID[pid]["is_control"] else plot.SERIES_PERSONA
    ax.scatter([i] * len(w), w, s=8, alpha=0.25, color=colour, edgecolors="none")
    ax.scatter([i], [w.mean()], s=70, color=colour, edgecolors=plot.SURFACE,
               linewidths=2, zorder=3)

ax.axhspan(lo, hi, color=plot.GRID, zorder=0)
ax.margins(y=0.12)   # keep the target band's edges visible
ax.set_xticks(range(len(order)), order, rotation=35, ha="right", fontsize=8.5)
ax.set_ylabel("words per generation")
ax.set_title(f"Generation length by persona\nband = {lo}–{hi} word target; "
             f"large dot = mean", color=plot.INK, fontsize=12, loc="left", pad=14)
fig.tight_layout()
plt.show()

## Emotion leakage

The instruction was to convey the emotion without naming it. This counts verbatim
occurrences only — synonyms are not detected, so treat it as a floor. Nothing is
dropped here or in `gen_data.py`; whether a leaked row is still a valid sample is
a call for you to make.

Read the **by-persona** breakdown, not just the overall rate. If leakage varies
systematically across personas, it varies along the x-axis of the headline figure,
and "probe accuracy falls with distance from the assistant" becomes hard to tell
apart from "the emotion word stops appearing in the text with distance". The
spread and the rank correlation below are the numbers that matter.

In [ ]:
df["names_emotion"] = [
    bool(re.search(rf"\b{re.escape(e)}\b", t, re.IGNORECASE))
    for e, t in zip(df["emotion"], df["text"])
]
n = int(df["names_emotion"].sum())
print(f"{n} of {len(df)} generations name their target emotion verbatim ({n/len(df):.2%})")

present = [p["id"] for p in personas.PERSONAS if (df["persona_id"] == p["id"]).any()]
by_persona = (df.groupby("persona_id")["names_emotion"]
                .agg(leaked="sum", n="count").reindex(present))
by_persona["rate"] = by_persona["leaked"] / by_persona["n"]
by_persona["intuited_rank"] = [personas.PERSONAS_BY_ID[p]["distance_rank"] for p in present]
by_persona["role"] = ["control" if personas.PERSONAS_BY_ID[p]["is_control"] else "persona"
                      for p in present]
display(by_persona.style.format({"rate": "{:.2%}"})
                  .background_gradient(cmap="Oranges", subset=["rate"]))

spread = by_persona["rate"].max() - by_persona["rate"].min()
print(f"spread across personas: {by_persona['rate'].min():.2%} – "
      f"{by_persona['rate'].max():.2%}  ({spread * 100:.1f}pp)")

pers_only = by_persona[by_persona["role"] != "control"]
if len(pers_only) > 2:
    r = pers_only["intuited_rank"].corr(pers_only["rate"], method="spearman")
    print(f"Spearman(intuited distance rank, leakage rate), personas only: {r:.3f}")
    print("far from 0 = leakage moves along the axis and is confounded with the "
          "headline relationship")

In [ ]:
# Detail: which (emotion, persona) cells the leakage actually sits in.
if n:
    display(df[df["names_emotion"]]
              .pivot_table(index="emotion", columns="persona_id",
                           values="scenario_id", aggfunc="count")
              .reindex(index=config.EMOTIONS, columns=present)
              .fillna(0).astype(int))
else:
    print("nothing leaked")

## Read the actual text

The one check nothing automated can do: does the persona sound like the persona,
and is the emotion actually present? Change `EMOTION` and re-run.

In [ ]:
EMOTION = "guilty"     # <- change me
SCENARIO = None        # <- or pin a scenario_id to hold the topic fixed

sel = df[df["emotion"] == EMOTION]
sel = sel[sel["scenario_id"] == (SCENARIO if SCENARIO is not None else sel["scenario_id"].iloc[0])]

print(f"emotion={EMOTION}  scenario={sel['scenario_id'].iloc[0]}")
print(f"topic: {sel['topic'].iloc[0]}\n")
for pid in [p["id"] for p in personas.PERSONAS]:
    row = sel[sel["persona_id"] == pid]
    if row.empty:
        continue
    print(f"--- {pid} " + "-" * (60 - len(pid)))
    print(row["text"].iloc[0], "\n")

---
**Next:** `python extract.py`, then `assistant_axis.py`, `probes.py`, `plot.py` —
results in `02_results.ipynb`.